# WiSARv1 Dataset Organization Investigation

This notebook performs a **read-only structural investigation** of a WiSARv1 dataset. It inventories paths, directory names, filenames, and metadata/manifests without opening image pixel data, modifying the dataset, copying files, extracting archives, resizing images, or training models.

The outputs are small aggregate text/CSV reports intended to answer whether defensible flight, recording-session, sequence, scene, or collection identifiers exist for leakage-safe splitting. Findings are labeled `documented`, `observed`, or `unknown`; different names are never treated as proof of different flights.

In [76]:
from __future__ import annotations

import csv
import os
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

# Set WISAR_DATASET_ROOT to the mounted dataset directory in Colab.
# Expected ZIP location: /content/drive/MyDrive/WiSARD/WiSARDv1.zip
DATASET_ROOT = Path(os.environ.get("WISAR_DATASET_ROOT", "/content/drive/MyDrive/WiSARD"))
if not DATASET_ROOT.exists() and Path("data/raw/WiSARD").is_dir():
    DATASET_ROOT = Path("data/raw/WiSARD")
REPORT_DIR = Path(os.environ.get("WISAR_REPORT_DIR", "results/dataset_audit"))
ZIP_NAME = "WiSARDv1.zip"
TREE_MAX_DEPTH = 4
MAX_METADATA_BYTES = 2_000_000
MAX_TEXT_LINES_PER_FILE = 2_000

METADATA_EXTENSIONS = {
    ".csv", ".json", ".xml", ".yaml", ".yml", ".txt", ".tsv", ".mat",
    ".ini", ".cfg", ".conf", ".toml", ".md", ".md5", ".log",
}
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".gif", ".ppm", ".pgm", ".dng", ".heic",
}
SEARCH_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "timestamp", "gps", "trajectory", "video", "mission",
)
GROUPING_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "mission", "run", "take", "trip", "set",
)


In [68]:
def tokenize_name(value: str) -> list[str]:
    """Split names into stable lowercase alphanumeric tokens without opening files."""
    return [token for token in re.split(r"[^a-zA-Z0-9]+", value.lower()) if token]


In [61]:
if not DATASET_ROOT.exists() or not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Set DATASET_ROOT to the mounted WiSARv1 directory; not found: {DATASET_ROOT}"
    )

root_resolved = DATASET_ROOT.resolve()
file_records = []
directory_records = []
metadata_records = []
extension_counts = Counter()
extension_by_directory = Counter()
directory_file_counts = Counter()
name_tokens = Counter()
directory_name_tokens = Counter()
tree_lines = [f"{root_resolved.name}/"]

# os.scandir reads directory entries and stat information; it does not decode image pixels.
stack = [(root_resolved, 0)]
while stack:
    current, depth = stack.pop()
    try:
        entries = sorted(os.scandir(current), key=lambda entry: (not entry.is_dir(follow_symlinks=False), entry.name.lower()))
    except (OSError, PermissionError) as error:
        directory_records.append({"relative_directory": str(current.relative_to(root_resolved)), "status": f"unreadable: {error}"})
        continue

    relative_current = current.relative_to(root_resolved)
    directory_records.append({"relative_directory": "." if relative_current == Path(".") else str(relative_current), "status": "read"})
    if depth <= TREE_MAX_DEPTH:
        tree_lines.extend([f"{'  ' * (depth + 1)}{'[D] ' if entry.is_dir(follow_symlinks=False) else '[F] '}{entry.name}" for entry in entries])

    for entry in entries:
        entry_path = Path(entry.path)
        relative_path = entry_path.relative_to(root_resolved)
        if entry.is_dir(follow_symlinks=False):
            directory_name_tokens.update(tokenize_name(entry.name))
            stack.append((entry_path, depth + 1))
            continue
        if not entry.is_file(follow_symlinks=False):
            continue

        suffix = entry_path.suffix.lower() or "[no_extension]"
        relative_directory = str(relative_path.parent)
        extension_counts[suffix] += 1
        extension_by_directory[(relative_directory, suffix)] += 1
        directory_file_counts[relative_directory] += 1
        tokens = tokenize_name(entry.name)
        name_tokens.update(tokens)
        record = {
            "relative_path": str(relative_path),
            "relative_directory": relative_directory,
            "filename": entry.name,
            "extension": suffix,
            "size_bytes": entry.stat(follow_symlinks=False).st_size,
            "name_tokens": ",".join(tokens),
        }
        file_records.append(record)
        if suffix in METADATA_EXTENSIONS:
            metadata_records.append(record.copy())

print(f"Dataset root: {root_resolved}")
print(f"Directories observed: {len(directory_records):,}")
print(f"Files observed: {len(file_records):,}")
print(f"Metadata-like files: {len(metadata_records):,}")
print("No image file was opened as pixel data.")

Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Directories observed: 1
Files observed: 1
Metadata-like files: 0
No image file was opened as pixel data.


In [77]:
zip_path = DATASET_ROOT / ZIP_NAME if DATASET_ROOT.is_dir() else None
zip_member_records = []
zip_metadata_matches = []
zip_metadata_snippets = []
zip_candidate_rows = []
zip_sensor_rows = []
zip_pattern_counts = Counter()
zip_pattern_examples = defaultdict(list)
zip_extension_counts = Counter()
zip_tree_lines = []
zip_status = "unknown"

if zip_path is not None and zip_path.is_file():
    zip_status = "observed"
    # ZipFile reads the archive directory and selected metadata members only; it never extracts files.
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        infos = archive.infolist()
        member_pairs = [
            (info, info.filename.replace("\\", "/").strip("/"))
            for info in infos
            if info.filename.strip("/")
        ]
        zip_member_records = []
        for info, member_path in member_pairs:
            if not member_path:
                continue
            suffix = Path(member_path).suffix.lower() or "[no_extension]"
            is_directory = info.is_dir() or info.filename.endswith(("/", "\\"))
            if not is_directory:
                zip_extension_counts[suffix] += 1
            zip_member_records.append({
                "member_path": member_path,
                "relative_directory": str(Path(member_path).parent),
                "filename": Path(member_path).name,
                "extension": suffix,
                "size_bytes": info.file_size,
                "compressed_size_bytes": info.compress_size,
                "is_directory": is_directory,
            })

        tree_nodes = {"": {"directories": set(), "files": set()}}
        for record in zip_member_records:
            parts = Path(record["member_path"]).parts
            for index in range(len(parts)):
                parent = "/".join(parts[:index])
                node = parts[index]
                tree_nodes.setdefault(parent, {"directories": set(), "files": set()})
                if index < len(parts) - 1 or record["is_directory"]:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["directories"].add(node)
                else:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["files"].add(node)

        zip_tree_lines = [f"{DATASET_ROOT.name}/{ZIP_NAME}"]
        for parent, node in sorted(tree_nodes.items()):
            depth = 0 if not parent else len(Path(parent).parts)
            if depth > TREE_MAX_DEPTH:
                continue
            prefix = "  " * (depth + 1)
            for directory in sorted(node["directories"]):
                zip_tree_lines.append(f"{prefix}[D] {directory}")
            for filename in sorted(node["files"]):
                zip_tree_lines.append(f"{prefix}[F] {filename}")
else:
    zip_status = "unknown"

print(f"ZIP path: {zip_path}")
print(f"ZIP status: {zip_status}")
print(f"ZIP members observed: {len(zip_member_records):,}")
print("ZIP extracted: no; image pixels opened: no")

ZIP path: None
ZIP status: unknown
ZIP members observed: 0
ZIP extracted: no; image pixels opened: no


In [63]:
metadata_matches = []
metadata_snippets = []
pattern_counts = Counter()
pattern_examples = defaultdict(list)

for record in metadata_records:
    path = root_resolved / record["relative_path"]
    filename_lower = record["filename"].lower()
    filename_terms = [term for term in SEARCH_TERMS if term in filename_lower]
    content_terms = []
    content = ""
    content_status = "not_read"
    if record["extension"] != ".mat" and record["size_bytes"] <= MAX_METADATA_BYTES:
        try:
            content = path.read_text(encoding="utf-8", errors="replace")
            content_status = "read"
        except (OSError, UnicodeError) as error:
            content_status = f"unreadable: {error}"
    elif record["extension"] == ".mat":
        content_status = "binary_mat_not_decoded"
    else:
        content_status = "skipped_over_size_limit"

    if content:
        content_lower = content.lower()
        content_terms = [term for term in SEARCH_TERMS if term in content_lower]
        for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
            line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
            if line_terms and len(metadata_snippets) < 200:
                metadata_snippets.append({
                    "relative_path": record["relative_path"],
                    "line_number": line_number,
                    "terms": ",".join(line_terms),
                    "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                })

    all_terms = sorted(set(filename_terms + content_terms))
    if all_terms:
        evidence_status = "documented" if content_terms else "observed"
        metadata_matches.append({
            "relative_path": record["relative_path"],
            "extension": record["extension"],
            "filename_terms": ",".join(filename_terms),
            "content_terms": ",".join(content_terms),
            "terms": ",".join(all_terms),
            "content_status": content_status,
            "evidence_status": evidence_status,
        })

for record in file_records:
    source_name = f"{record['relative_directory']}/{record['filename']}"
    for pattern, label in (
        (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
        (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
        (r"[A-Za-z]+[_-]\d+", "label_number"),
    ):
        matches = re.findall(pattern, source_name, flags=re.IGNORECASE)
        if matches:
            pattern_counts[label] += len(matches)
            for match in matches[:3]:
                if len(pattern_examples[label]) < 10:
                    pattern_examples[label].append(match.strip(" _-"))

candidate_rows = []
for record in file_records:
    path_parts = Path(record["relative_path"]).parts
    for part in path_parts:
        part_lower = part.lower()
        matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
        has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
        if matched_terms or has_identifier_shape:
            candidate_rows.append({
                "candidate": part,
                "source_path": record["relative_path"],
                "matched_terms": ",".join(matched_terms),
                "evidence_status": "observed",
                "interpretation": "Naming/path pattern only; not proof of an independent flight or session.",
            })

# Canonicalize RGB/thermal paths only for comparison; this does not alter source paths.
sensor_groups = defaultdict(lambda: {"rgb": set(), "thermal": set()})
for record in file_records:
    parts = list(Path(record["relative_path"]).parts)
    sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
    if not sensors:
        continue
    sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
    canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
    canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
    sensor_groups[canonical][sensor].add(record["filename"].lower())

sensor_rows = []
for canonical, groups in sorted(sensor_groups.items()):
    has_both = bool(groups["rgb"] and groups["thermal"])
    sensor_rows.append({
        "canonical_path_without_sensor": canonical,
        "rgb_file_count": len(groups["rgb"]),
        "thermal_file_count": len(groups["thermal"]),
        "shared_filename_count": len(groups["rgb"] & groups["thermal"]),
        "evidence_status": "observed" if has_both else "unknown",
        "interpretation": (
            "RGB and thermal occur under a shared canonical path; verify timestamps/metadata before grouping."
            if has_both else
            "No paired path observed here; this does not prove different flights or sessions."
        ),
    })

print(f"Metadata files matching investigation terms: {len(metadata_matches):,}")
print(f"Candidate grouping path/name observations: {len(candidate_rows):,}")
print(f"RGB/thermal canonical groups: {len(sensor_rows):,}")

Metadata files matching investigation terms: 0
Candidate grouping path/name observations: 0
RGB/thermal canonical groups: 0


In [70]:
if zip_status == "observed":
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        info_by_path = {info.filename.replace("\\", "/").strip("/"): info for info in archive.infolist()}
        for record in zip_member_records:
            member_path = record["member_path"]
            path_lower = member_path.lower()
            filename_terms = [term for term in SEARCH_TERMS if term in path_lower]
            content_terms = []
            content_status = "not_read"
            content = ""
            info = info_by_path.get(member_path)
            is_safe_metadata = record["extension"] in METADATA_EXTENSIONS and record["extension"] not in IMAGE_EXTENSIONS
            if info and not record["is_directory"] and is_safe_metadata and info.file_size <= MAX_METADATA_BYTES:
                try:
                    with archive.open(info, mode="r") as metadata_handle:
                        content = metadata_handle.read(MAX_METADATA_BYTES).decode("utf-8", errors="replace")
                    content_status = "read_from_zip_without_extraction"
                except (OSError, RuntimeError, UnicodeError) as error:
                    content_status = f"unreadable: {error}"
            elif record["extension"] == ".mat":
                content_status = "binary_mat_not_decoded"
            elif record["is_directory"]:
                content_status = "directory_member"
            elif record["extension"] in METADATA_EXTENSIONS:
                content_status = "skipped_over_size_limit"

            if content:
                content_terms = [term for term in SEARCH_TERMS if term in content.lower()]
                for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
                    line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
                    if line_terms and len(zip_metadata_snippets) < 200:
                        zip_metadata_snippets.append({
                            "member_path": member_path,
                            "line_number": line_number,
                            "terms": ",".join(line_terms),
                            "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                        })

            all_terms = sorted(set(filename_terms + content_terms))
            if all_terms:
                zip_metadata_matches.append({
                    "member_path": member_path,
                    "extension": record["extension"],
                    "filename_terms": ",".join(filename_terms),
                    "content_terms": ",".join(content_terms),
                    "terms": ",".join(all_terms),
                    "content_status": content_status,
                    "evidence_status": "documented" if content_terms else "observed",
                })

            for pattern, label in (
                (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
                (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
                (r"[A-Za-z]+[_-]\d+", "label_number"),
            ):
                matches = re.findall(pattern, member_path, flags=re.IGNORECASE)
                if matches:
                    zip_pattern_counts[label] += len(matches)
                    for match in matches[:3]:
                        if len(zip_pattern_examples[label]) < 10:
                            zip_pattern_examples[label].append(match.strip(" _-"))

            for part in Path(member_path).parts:
                part_lower = part.lower()
                matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
                has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
                if matched_terms or has_identifier_shape:
                    zip_candidate_rows.append({
                        "candidate": part,
                        "source_path": member_path,
                        "matched_terms": ",".join(matched_terms),
                        "evidence_status": "observed",
                        "interpretation": "ZIP path/name pattern only; not proof of an independent flight or session.",
                    })

            parts = list(Path(member_path).parts)
            sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
            if sensors and not record["is_directory"]:
                sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
                canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
                canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
                existing = next((row for row in zip_sensor_rows if row["canonical_path_without_sensor"] == canonical), None)
                if existing is None:
                    existing = {
                        "canonical_path_without_sensor": canonical,
                        "rgb_filenames": set(),
                        "thermal_filenames": set(),
                    }
                    zip_sensor_rows.append(existing)
                existing[f"{sensor}_filenames"].add(record["filename"].lower())

    for row in zip_sensor_rows:
        rgb_names = row.pop("rgb_filenames")
        thermal_names = row.pop("thermal_filenames")
        row.update({
            "rgb_file_count": len(rgb_names),
            "thermal_file_count": len(thermal_names),
            "shared_filename_count": len(rgb_names & thermal_names),
            "evidence_status": "observed" if rgb_names and thermal_names else "unknown",
            "interpretation": (
                "RGB and thermal occur under a shared canonical ZIP path; verify timestamps/metadata before grouping."
                if rgb_names and thermal_names else
                "No paired ZIP path observed; this does not prove different flights or sessions."
            ),
        })

print(f"ZIP metadata files matching investigation terms: {len(zip_metadata_matches):,}")
print(f"ZIP candidate grouping path/name observations: {len(zip_candidate_rows):,}")
print(f"ZIP RGB/thermal canonical groups: {len(zip_sensor_rows):,}")

ZIP metadata files matching investigation terms: 1
ZIP candidate grouping path/name observations: 5
ZIP RGB/thermal canonical groups: 0


In [64]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def write_csv(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    output_path = REPORT_DIR / filename
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

extension_rows = [
    {"extension": extension, "file_count": count}
    for extension, count in sorted(extension_counts.items(), key=lambda item: (-item[1], item[0]))
]
directory_extension_rows = [
    {"relative_directory": directory, "extension": extension, "file_count": count}
    for (directory, extension), count in sorted(extension_by_directory.items())
]
directory_rows = [
    {"relative_directory": directory, "file_count": count}
    for directory, count in sorted(directory_file_counts.items())
]
pattern_rows = [
    {"pattern_type": label, "match_count": pattern_counts[label], "examples": "; ".join(pattern_examples[label])}
    for label in sorted(pattern_counts)
]

write_csv("extension_counts.csv", extension_rows, ["extension", "file_count"])
write_csv("directory_extension_counts.csv", directory_extension_rows, ["relative_directory", "extension", "file_count"])
write_csv("directory_file_counts.csv", directory_rows, ["relative_directory", "file_count"])
write_csv("metadata_term_matches.csv", metadata_matches, ["relative_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
write_csv("metadata_term_snippets.csv", metadata_snippets, ["relative_path", "line_number", "terms", "snippet"])
write_csv("naming_patterns.csv", pattern_rows, ["pattern_type", "match_count", "examples"])
write_csv("grouping_candidates.csv", candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
write_csv("rgb_thermal_grouping.csv", sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])

(REPORT_DIR / "directory_tree.txt").write_text("\n".join(tree_lines) + "\n", encoding="utf-8")

reported_documented = sum(row["evidence_status"] == "documented" for row in metadata_matches)
reported_observed = sum(row["evidence_status"] == "observed" for row in metadata_matches) + len(candidate_rows)
summary = f"""WiSARv1 organization investigation
=================================
Dataset root: {root_resolved}
Files observed: {len(file_records):,}
Directories observed: {len(directory_records):,}
Metadata-like files observed: {len(metadata_records):,}
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {reported_documented:,} metadata files contain search terms
observed: {reported_observed:,} filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports
-------
{chr(10).join(sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))}
"""
(REPORT_DIR / "organization_investigation_report.txt").write_text(summary, encoding="utf-8")

print(summary)


WiSARv1 organization investigation
Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Files observed: 1
Directories observed: 1
Metadata-like files observed: 0
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: 0 metadata files contain search terms
observed: 0 filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports


In [65]:
zip_report_dir = REPORT_DIR
zip_report_dir.mkdir(parents=True, exist_ok=True)

if zip_status == "observed":
    zip_extension_rows = [
        {"extension": extension, "file_count": count}
        for extension, count in sorted(zip_extension_counts.items(), key=lambda item: (-item[1], item[0]))
    ]
    zip_pattern_rows = [
        {"pattern_type": label, "match_count": zip_pattern_counts[label], "examples": "; ".join(zip_pattern_examples[label])}
        for label in sorted(zip_pattern_counts)
    ]
    write_csv("zip_extension_counts.csv", zip_extension_rows, ["extension", "file_count"])
    write_csv("zip_metadata_term_matches.csv", zip_metadata_matches, ["member_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
    write_csv("zip_metadata_term_snippets.csv", zip_metadata_snippets, ["member_path", "line_number", "terms", "snippet"])
    write_csv("zip_naming_patterns.csv", zip_pattern_rows, ["pattern_type", "match_count", "examples"])
    write_csv("zip_grouping_candidates.csv", zip_candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
    write_csv("zip_rgb_thermal_grouping.csv", zip_sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])
    (zip_report_dir / "zip_directory_tree.txt").write_text("\n".join(zip_tree_lines) + "\n", encoding="utf-8")

    zip_documented = sum(row["evidence_status"] == "documented" for row in zip_metadata_matches)
    zip_observed = sum(row["evidence_status"] == "observed" for row in zip_metadata_matches) + len(zip_candidate_rows)
    zip_summary = f"""WiSARDv1 ZIP organization investigation
=======================================
ZIP path: {zip_path}
ZIP members observed: {len(zip_member_records):,}
Image pixels opened: no
ZIP extracted/copied/moved/modified: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {zip_documented:,} ZIP metadata/path records with metadata text terms
observed: {zip_observed:,} ZIP filename/content/path observations
unknown: different ZIP folder names are not treated as different flights without documentation or metadata

Metadata reads
--------------
Only non-image metadata-like members at or below MAX_METADATA_BYTES were read with ZipFile.open().
No image member was opened, decoded, resized, or extracted.

RGB/thermal
-----------
Shared canonical ZIP paths are labeled observed correspondence only.
Non-overlap is labeled unknown, not evidence of different flights or sessions.
"""
    (zip_report_dir / "zip_organization_investigation_report.txt").write_text(zip_summary, encoding="utf-8")
    print(zip_summary)
else:
    print("No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.")


No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.


# Collection-Context Grouping Analysis

**Purpose:** construct a conservative, reproducible grouping unit for future leakage-safe dataset splitting.

**Terminology:**
- **Recording** means one observed top-level acquisition/stream folder in the ZIP.
- **Collection context** means a candidate higher-level grouping derived from the common prefix of related recording names.
- **Flight** is not used unless external dataset documentation or metadata explicitly supports that identity.
- A collection context is therefore an **observed grouping candidate**, not a documented flight identity.

Individual frames are not independent observations for splitting. Sequential frames from one recording must remain together, and related VIS and IR recordings sharing a candidate context should remain together for leakage control. Different names alone do not prove different flights, and matching stream identifiers alone do not prove temporal synchronization. This notebook makes no train/validation/test assignment.

The observed naming pattern is:

```text
210417_MtErie_Enterprise_VIS_0003
210417_MtErie_Enterprise_VIS_0005
210417_MtErie_Enterprise_IR_0004
210417_MtErie_Enterprise_IR_0006
candidate collection context: 210417_MtErie_Enterprise

210327_Airfield_FLIR_VIS_1
210327_Airfield_FLIR_IR_1
candidate collection context: 210327_Airfield_FLIR
```

These are naming/path observations only. The parser uses exactly: `date_token + "_" + context_prefix + "_" + modality_token + "_" + numeric_stream_identifier`, where the date is six digits at the beginning, modality is exactly `VIS` or `IR`, the identifier is digits at the end, and the collection context is the unchanged prefix before `VIS`/`IR`. Names that do not satisfy this rule remain explicit ambiguous/unparsed recording folders and are never silently merged or discarded.

In [85]:
ANNOTATION_EXTENSIONS = {".txt", ".xml", ".json", ".csv", ".tsv", ".mat", ".yaml", ".yml", ".ann", ".label", ".labels"}
MODALITY_TOKENS = {"VIS": "VIS", "IR": "IR"}
FRAME_INDEX_PATTERNS = (
    re.compile(r"(?i)(?:frame|image|img|rgb|ir|thermal)[_-]?(\d+)(?:\D*)$"),
    re.compile(r"(?:^|[_-])(\d{3,})(?:\D*)$"),
)


def parse_recording_name(recording_name: str) -> dict:
    tokens = recording_name.split("_")
    positions = [index for index, token in enumerate(tokens) if token in MODALITY_TOKENS]
    result = {
        "recording_name": recording_name, "collection_context": "", "modality": "",
        "date_token": tokens[0] if tokens and re.fullmatch(r"\d{6}", tokens[0]) else "",
        "location_token": "", "platform_token": "",
        "stream_identifier": tokens[-1] if tokens and tokens[-1].isdigit() else "",
        "parsing_status": "ambiguous_or_unparsed",
    }
    if len(positions) != 1:
        return result
    modality_index = positions[0]
    valid = (
        bool(result["date_token"]) and modality_index >= 2
        and bool(re.fullmatch(r"\d+", tokens[-1]))
        and modality_index == len(tokens) - 2
    )
    if not valid:
        return result
    prefix = tokens[:modality_index]
    result.update({
        "collection_context": "_".join(prefix), "modality": tokens[modality_index],
        "location_token": "_".join(prefix[1:-1]), "platform_token": prefix[-1],
        "parsing_status": "parsed",
    })
    return result


def top_level_recording_component(member_path: str) -> tuple[str, str]:
    parts = Path(member_path).parts
    return (parts[0], parts[0]) if len(parts) >= 2 else ("", "")


def frame_index_from_name(filename: str) -> int | None:
    for pattern in FRAME_INDEX_PATTERNS:
        match = pattern.search(Path(filename).stem)
        if match:
            return int(match.group(1))
    return None


def is_image_member(record: dict) -> bool:
    return not record["is_directory"] and record["extension"] in IMAGE_EXTENSIONS


def is_annotation_member(record: dict) -> bool:
    return not record["is_directory"] and record["extension"] in ANNOTATION_EXTENSIONS


collection_context_summary = []
recording_stream_summary = []
collection_context_modalities = []
recording_sequence_structure = []
collection_context_sanity_checks = []
collection_context_ambiguities = []

if zip_status == "observed":
    recording_aggregates = {}
    relative_path_counts = Counter()
    recording_name_to_paths = defaultdict(set)
    contexts = defaultdict(lambda: {"recordings": [], "VIS": [], "IR": [], "visual_images": 0, "thermal_images": 0, "annotation_count": 0, "dates": set(), "locations": set(), "platforms": set(), "identifiers": set()})

    for record in sorted(zip_member_records, key=lambda item: item["member_path"]):
        member_path = record["member_path"]
        relative_path_counts[member_path] += 1
        recording_name, recording_path = top_level_recording_component(member_path)
        if not recording_name:
            continue
        aggregate = recording_aggregates.setdefault(recording_path, {
            **parse_recording_name(recording_name), "recording_path": recording_path,
            "image_count": 0, "annotation_count": 0, "image_extensions": Counter(),
            "annotation_extensions": Counter(), "example_frame_names": [], "frame_indices": set(),
            "frame_index_parseable": True,
        })
        recording_name_to_paths[recording_name].add(recording_path)
        if record["is_directory"]:
            continue
        if is_image_member(record):
            aggregate["image_count"] += 1
            aggregate["image_extensions"][record["extension"]] += 1
            if len(aggregate["example_frame_names"]) < 5:
                aggregate["example_frame_names"].append(Path(member_path).name)
            index = frame_index_from_name(Path(member_path).name)
            if index is None:
                aggregate["frame_index_parseable"] = False
            else:
                aggregate["frame_indices"].add(index)
        elif is_annotation_member(record):
            aggregate["annotation_count"] += 1
            aggregate["annotation_extensions"][record["extension"]] += 1

    for recording_path, aggregate in sorted(recording_aggregates.items()):
        parsed = aggregate["parsing_status"] == "parsed"
        if parsed:
            context = contexts[aggregate["collection_context"]]
            context["recordings"].append(aggregate)
            context[aggregate["modality"]].append(aggregate)
            context["visual_images"] += aggregate["image_count"] if aggregate["modality"] == "VIS" else 0
            context["thermal_images"] += aggregate["image_count"] if aggregate["modality"] == "IR" else 0
            context["annotation_count"] += aggregate["annotation_count"]
            for field in ("dates", "locations", "platforms", "identifiers"):
                context[field].add(aggregate[{"dates": "date_token", "locations": "location_token", "platforms": "platform_token", "identifiers": "stream_identifier"}[field]])
        else:
            reason = "unrecognized_modality_or_recording_name_pattern"
            if not aggregate["date_token"]:
                reason = "recording_name_did_not_begin_with_six_digit_date"
            elif not aggregate["modality"]:
                reason = "recording_name_did_not_contain_exact_VIS_or_IR_token"
            elif not aggregate["stream_identifier"]:
                reason = "recording_name_did_not_end_with_numeric_stream_identifier"
            collection_context_ambiguities.append({
                "recording_name": aggregate["recording_name"], "recording_path": recording_path,
                "reason": reason, "evidence_status": "observed",
                "interpretation": "One ambiguous recording-folder row; no collection context was assigned.",
            })

        indices = aggregate["frame_indices"]
        parse_status = "reliably_parseable" if aggregate["frame_index_parseable"] and indices else "unknown_or_mixed"
        minimum = min(indices) if parse_status == "reliably_parseable" else ""
        maximum = max(indices) if parse_status == "reliably_parseable" else ""
        gaps = sorted(set(range(minimum, maximum + 1)) - indices) if parse_status == "reliably_parseable" else []
        recording_stream_summary.append({
            "recording_name": aggregate["recording_name"], "recording_path": recording_path,
            "collection_context": aggregate["collection_context"], "modality": aggregate["modality"],
            "date_token": aggregate["date_token"], "location_token": aggregate["location_token"],
            "platform_token": aggregate["platform_token"], "stream_identifier": aggregate["stream_identifier"],
            "image_count": aggregate["image_count"], "annotation_count": aggregate["annotation_count"],
            "image_extensions": ";".join(sorted(aggregate["image_extensions"])),
            "annotation_extensions": ";".join(sorted(aggregate["annotation_extensions"])),
            "minimum_frame_index": minimum, "maximum_frame_index": maximum,
            "unique_frame_index_count": len(indices), "obvious_gap_count": len(gaps),
            "obvious_gap_examples": ";".join(str(value) for value in gaps[:20]),
            "example_frame_names": ";".join(sorted(aggregate["example_frame_names"])),
            "parsing_status": aggregate["parsing_status"], "evidence_status": "observed",
            "interpretation": "Recording identity, modality, grouping, and frame structure are ZIP path/name observations only; no flight or synchronization claim is made.",
        })
        recording_sequence_structure.append({
            "recording_name": aggregate["recording_name"], "recording_path": recording_path,
            "collection_context": aggregate["collection_context"], "frame_index_parse_status": parse_status,
            "minimum_frame_index": minimum, "maximum_frame_index": maximum,
            "unique_frame_index_count": len(indices), "obvious_gap_count": len(gaps),
            "obvious_gap_examples": ";".join(str(value) for value in gaps[:20]),
            "evidence_status": "observed",
            "interpretation": "Sequential frame structure is inferred from filenames only; no temporal timestamps or synchronization are claimed.",
        })

    def add_check(check_type: str, status: str, details: str, interpretation: str) -> None:
        collection_context_sanity_checks.append({"check_type": check_type, "status": status, "details": details, "evidence_status": "observed", "interpretation": interpretation})

    duplicate_paths = {path: count for path, count in relative_path_counts.items() if count > 1}
    add_check("duplicate_recording_paths", "flagged" if duplicate_paths else "clear", str(duplicate_paths) if duplicate_paths else "No duplicate ZIP member paths observed.", "Diagnostic only; duplicates are not merged or discarded.")
    duplicate_names = {name: sorted(paths) for name, paths in recording_name_to_paths.items() if len(paths) > 1}
    add_check("duplicate_recording_names", "flagged" if duplicate_names else "clear", str(duplicate_names) if duplicate_names else "No duplicate recording names observed.", "Full recording paths remain the audit identity.")
    add_check("recording_in_multiple_contexts", "clear", "A recording folder has one deterministic parsed context or none.", "Diagnostic only; no automatic merging is performed.")
    for context_name, context in sorted(contexts.items()):
        if len(context["VIS"]) > 1:
            add_check("multiple_VIS_recordings_in_context", "observed", f"{context_name}: {len(context['VIS'])} VIS recordings", "Multiple VIS streams remain in one observed candidate context.")
        if len(context["IR"]) > 1:
            add_check("multiple_IR_recordings_in_context", "observed", f"{context_name}: {len(context['IR'])} IR recordings", "Multiple IR streams remain in one observed candidate context.")
        if context["VIS"] and context["IR"]:
            add_check("context_contains_both_VIS_and_IR", "observed", f"{context_name}: VIS and IR recording folders coexist", "This does not prove pairing, synchronization, camera identity, or one flight.")
    names = sorted(contexts)
    similar = [f"{left} ~ {right}" for index, left in enumerate(names) for right in names[index + 1:] if left.rstrip("_0123456789") == right.rstrip("_0123456789")]
    add_check("suspiciously_similar_context_names", "flagged" if similar else "clear", "; ".join(similar) if similar else "No suspiciously similar context names observed.", "Similar names are diagnostics only; contexts are not automatically merged.")
    add_check("ambiguous_recording_names", "flagged" if collection_context_ambiguities else "clear", f"{len(collection_context_ambiguities)} unique ambiguous/unparsed recording folders", "Ambiguous folders remain explicit and unassigned.")
    unrecognized = sum(row["modality"] == "" for row in recording_stream_summary)
    add_check("unrecognized_modality_naming", "flagged" if unrecognized else "clear", f"{unrecognized} recording folders lack exact VIS/IR parsing", "Only exact VIS and IR tokens are recognized.")
    gap_rows = [row for row in recording_sequence_structure if row["obvious_gap_count"]]
    add_check("frame_index_gaps", "flagged" if gap_rows else "clear", f"{len(gap_rows)} recording folders have filename-index gaps", "Gaps are filename diagnostics only.")

    for context_name, context in sorted(contexts.items()):
        modalities = sorted({item["modality"] for item in context["recordings"]})
        collection_context_summary.append({
            "collection_context": context_name, "top_level_recording_count": len(context["recordings"]),
            "visual_recording_count": len(context["VIS"]), "thermal_recording_count": len(context["IR"]),
            "visual_image_count": context["visual_images"], "thermal_image_count": context["thermal_images"],
            "annotation_count": context["annotation_count"],
            "recording_names": ";".join(sorted(item["recording_name"] for item in context["recordings"])),
            "modality_set": ";".join(modalities), "date_token": ";".join(sorted(value for value in context["dates"] if value)),
            "location_token": ";".join(sorted(value for value in context["locations"] if value)),
            "platform_token": ";".join(sorted(value for value in context["platforms"] if value)),
            "identifier_tokens": ";".join(sorted(value for value in context["identifiers"] if value)),
            "parsing_status": "parsed", "evidence_status": "observed",
            "interpretation": "Candidate context from an unchanged recording-name prefix; not a documented flight identity.",
        })
        collection_context_modalities.append({
            "collection_context": context_name, "contains_both_modalities": "yes" if context["VIS"] and context["IR"] else "no",
            "visual_stream_count": len(context["VIS"]), "thermal_stream_count": len(context["IR"]),
            "visual_image_count": context["visual_images"], "thermal_image_count": context["thermal_images"],
            "evidence_status": "observed",
            "interpretation": "VIS/IR coexistence does not mean synchronized, one-to-one paired, same-camera, or one-flight recordings.",
        })

    recording_stream_summary.sort(key=lambda row: row["recording_path"])
    recording_sequence_structure.sort(key=lambda row: row["recording_path"])
    collection_context_ambiguities.sort(key=lambda row: row["recording_path"])
    print(f"Candidate collection contexts: {len(collection_context_summary):,}")
    print(f"Unique recording folders: {len(recording_aggregates):,}")
    print(f"Parsed recording folders: {sum(row['parsing_status'] == 'parsed' for row in recording_stream_summary):,}")
    print(f"VIS recordings: {sum(row['modality'] == 'VIS' for row in recording_stream_summary):,}")
    print(f"IR recordings: {sum(row['modality'] == 'IR' for row in recording_stream_summary):,}")
    print(f"Contexts containing both modalities: {sum(row['contains_both_modalities'] == 'yes' for row in collection_context_modalities):,}")
    print(f"Ambiguous/unparsed recording folders: {len(collection_context_ambiguities):,}")
    print("No train/validation/test split created.")
else:
    print("ZIP collection-context analysis skipped because WiSARDv1.zip was not detected.")

ZIP collection-context analysis skipped because WiSARDv1.zip was not detected.


In [79]:
if zip_status == "observed":
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    write_csv(
        "collection_context_summary.csv",
        collection_context_summary,
        [
            "collection_context", "top_level_recording_count", "visual_recording_count",
            "thermal_recording_count", "visual_image_count", "thermal_image_count",
            "annotation_count", "recording_names", "modality_set", "date_token",
            "location_token", "platform_token", "identifier_tokens", "parsing_status",
            "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "recording_stream_summary.csv",
        recording_stream_summary,
        [
            "recording_name", "recording_path", "collection_context", "modality", "date_token",
            "location_token", "platform_token", "stream_identifier", "image_count",
            "annotation_count", "image_extensions", "annotation_extensions", "minimum_frame_index",
            "maximum_frame_index", "unique_frame_index_count", "obvious_gap_count",
            "obvious_gap_examples", "example_frame_names", "parsing_status", "evidence_status",
            "interpretation",
        ],
    )
    write_csv(
        "collection_context_modalities.csv",
        collection_context_modalities,
        [
            "collection_context", "contains_both_modalities", "visual_stream_count",
            "thermal_stream_count", "visual_image_count", "thermal_image_count",
            "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "recording_sequence_structure.csv",
        recording_sequence_structure,
        [
            "recording_name", "recording_path", "collection_context", "frame_index_parse_status",
            "minimum_frame_index", "maximum_frame_index", "unique_frame_index_count",
            "obvious_gap_count", "obvious_gap_examples", "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "collection_context_ambiguities.csv",
        collection_context_ambiguities,
        ["recording_name", "recording_path", "reason", "evidence_status", "interpretation"],
    )
    write_csv(
        "collection_context_sanity_checks.csv",
        collection_context_sanity_checks,
        ["check_type", "status", "details", "evidence_status", "interpretation"],
    )

    both_modality_contexts = [
        row["collection_context"] for row in collection_context_modalities
        if row["contains_both_modalities"] == "yes"
    ]
    top_contexts = sorted(
        collection_context_summary,
        key=lambda row: (-row["visual_image_count"] - row["thermal_image_count"], row["collection_context"]),
    )
    context_lines = [
        f"- {row['collection_context']}: total_images={row['visual_image_count'] + row['thermal_image_count']}, "
        f"VIS={row['visual_recording_count']}, IR={row['thermal_recording_count']}"
        for row in top_contexts
    ]
    flagged_checks = [row for row in collection_context_sanity_checks if row["status"] == "flagged"]
    parsed_recording_count = sum(row["parsing_status"] == "parsed" for row in recording_stream_summary)
    visual_recording_count = sum(row["modality"] == "VIS" for row in recording_stream_summary)
    thermal_recording_count = sum(row["modality"] == "IR" for row in recording_stream_summary)
    total_visual_images = sum(row["image_count"] for row in recording_stream_summary if row["modality"] == "VIS")
    total_thermal_images = sum(row["image_count"] for row in recording_stream_summary if row["modality"] == "IR")
    collection_context_report = f"""WiSARDv1 collection-context grouping analysis
===============================================
Dataset/ZIP path: {zip_path}
ZIP members: {len(zip_member_records):,}
Unique recording folders: {len(recording_aggregates):,}
Parsed recording folders: {parsed_recording_count:,}
Ambiguous recording folders: {len(collection_context_ambiguities):,}
Candidate collection contexts: {len(collection_context_summary):,}
VIS recordings: {visual_recording_count:,}
IR recordings: {thermal_recording_count:,}
Contexts containing both VIS and IR: {len(both_modality_contexts):,}

Top collection contexts ranked by total image count
----------------------------------------------------
{chr(10).join(context_lines) if context_lines else '- none observed'}

Explicit grouping rule
----------------------
A recording_name must have the conceptual form date_token_context_prefix_modality_token_numeric_stream_identifier.
date_token is exactly six digits at the beginning; modality_token is exactly VIS or IR; numeric_stream_identifier
is digits at the end; collection_context is the exact unchanged prefix before VIS/IR. The first ZIP directory
component is preserved as the observed recording folder and full recording_path. No token meaning is inferred
beyond its observed position/name structure.

Methodological interpretation
-----------------------------
Collection contexts are conservative grouping candidates derived from observed recording-name structure.
They are potential leakage-control units, not documented flight identities. No claim of flight independence
or temporal synchronization is made solely from naming structure. Individual frames are not independent
splitting units; sequential frames and related VIS/IR recordings sharing a candidate context should remain
together for later leakage-safe design. Matching stream identifiers do not prove paired frames.

Sanity checks
-------------
Flagged diagnostic checks: {len(flagged_checks):,}
{chr(10).join(f"- {row['check_type']}: {row['details']}" for row in flagged_checks) if flagged_checks else '- none'}
Ambiguous recording folders are listed once each in collection_context_ambiguities.csv. Frame gaps are
filename diagnostics only. VIS/IR coexistence does not prove synchronization, pairing, same camera, or one flight.

Safety and reproducibility
--------------------------
The ZIP central directory was inspected directly with Python zipfile.ZipFile. No extraction, image opening,
decoding, resizing, transformation, copying, moving, deletion, model training, or split assignment occurred.
Outputs are sorted deterministically by path/name. No train/validation/test split was created.
"""
    (REPORT_DIR / "collection_context_analysis_report.txt").write_text(collection_context_report, encoding="utf-8")
    print(collection_context_report)
else:
    print("Collection-context reports skipped because WiSARDv1.zip was not detected.")

Collection-context reports skipped because WiSARDv1.zip was not detected.


In [86]:
parser_examples = {
    "210417_MtErie_Enterprise_VIS_0003": ("210417_MtErie_Enterprise", "VIS", "0003"),
    "210417_MtErie_Enterprise_IR_0004": ("210417_MtErie_Enterprise", "IR", "0004"),
    "210327_Airfield_FLIR_VIS_1": ("210327_Airfield_FLIR", "VIS", "1"),
    "210327_Airfield_FLIR_IR_1": ("210327_Airfield_FLIR", "IR", "1"),
}
for example_name, expected in parser_examples.items():
    parsed_example = parse_recording_name(example_name)
    assert (
        parsed_example["collection_context"],
        parsed_example["modality"],
        parsed_example["stream_identifier"],
    ) == expected
    assert parsed_example["parsing_status"] == "parsed"

if zip_status == "observed":
    ambiguity_keys = [
        (row["recording_name"], row["recording_path"])
        for row in collection_context_ambiguities
    ]
    assert len(ambiguity_keys) == len(set(ambiguity_keys))
    print(f"Parser self-check passed for {len(parser_examples)} supplied examples.")
    print("Ambiguity rows are unique at recording-folder level.")
else:
    print(f"Parser self-check passed for {len(parser_examples)} supplied examples.")
    print("Ambiguity uniqueness check deferred until WiSARDv1.zip is available.")

Parser self-check passed for 4 supplied examples.
Ambiguity uniqueness check deferred until WiSARDv1.zip is available.


In [87]:
if zip_status == "observed":
    print(f"Candidate collection contexts: {len(collection_context_summary)}")
    print(f"Unique recording folders: {len(recording_aggregates)}")
    print(f"Parsed recording folders: {sum(row['parsing_status'] == 'parsed' for row in recording_stream_summary)}")
    print(f"VIS recordings: {sum(row['modality'] == 'VIS' for row in recording_stream_summary)}")
    print(f"IR recordings: {sum(row['modality'] == 'IR' for row in recording_stream_summary)}")
    print(f"Contexts containing both modalities: {sum(row['contains_both_modalities'] == 'yes' for row in collection_context_modalities)}")
    print(f"Ambiguous/unparsed recording folders: {len(collection_context_ambiguities)}")
    print(f"Total visual images: {sum(row['image_count'] for row in recording_stream_summary if row['modality'] == 'VIS')}")
    print(f"Total thermal images: {sum(row['image_count'] for row in recording_stream_summary if row['modality'] == 'IR')}")
else:
    print("Candidate collection contexts: unavailable (WiSARDv1.zip not detected)")
    print("Unique recording folders: unavailable")
    print("Parsed recording folders: unavailable")
    print("VIS recordings: unavailable")
    print("IR recordings: unavailable")
    print("Contexts containing both modalities: unavailable")
    print("Ambiguous/unparsed recording folders: unavailable")
    print("Total visual images: unavailable")
    print("Total thermal images: unavailable")
print("No train/validation/test split created.")
print("Next methodological step: inspect the collection-context tables and decide the frozen group-level split.")

Candidate collection contexts: unavailable (WiSARDv1.zip not detected)
Unique recording folders: unavailable
Parsed recording folders: unavailable
VIS recordings: unavailable
IR recordings: unavailable
Contexts containing both modalities: unavailable
Ambiguous/unparsed recording folders: unavailable
Total visual images: unavailable
Total thermal images: unavailable
No train/validation/test split created.
Next methodological step: inspect the collection-context tables and decide the frozen group-level split.
